# Continuar v5 desde 200 de 240 corridas

Conectar como dataset privado el contenido extraído de **results_closure_v5 (1).zip**, junto con `arjuntejaswi/plant-village` y `warcoder/potato-leaf-disease-dataset`. No depende del nombre de la cuenta de Kaggle. Iniciar una sesión nueva con GPU e Internet.

El estado auditado contiene 200 corridas completas y un checkpoint de ViT tras la época 8. Faltan 40 corridas: 25 de campo y 15 agrupadas de laboratorio, todas de ViT. Se conserva el checkpoint para comenzar por la época 9 de la corrida pendiente. Una época parcialmente ejecutada se repite.

El detector exige al menos 200 marcadores y selecciona un único estado compatible; si hay varios, solicita definir `RESUME_DIR` en la configuración. Luego el motor verifica la integridad real de las predicciones y particiones. No se reinician las corridas completas.

Protocolo conservado: 240 corridas, cuatro modelos, dos conjuntos, dos particiones y quince semillas; diez épocas. Las 60 corridas aleatorias originales de laboratorio se conservan con predicciones idénticas. Laboratorio: 2.152 imágenes y 538 identidades de hoja publicadas. Campo: 1.296 imágenes de una sola fuente, clases Healthy/Fungi/Phytophthora y agrupamiento DINO exploratorio. La validación humana de los grupos de campo sigue pendiente. La corrección se diseñó después de los resultados v4.2 y no está prerregistrada.


In [ ]:
# Ejecutar en una sesión nueva para evitar paquetes importados de otras versiones.
import sys, subprocess, importlib.metadata, time
print("DEPENDENCIAS: comprobando versiones del experimento original...", flush=True)
required={'timm':'1.0.26','ImageHash':'4.3.2','numpy':'2.0.2','pandas':'2.3.3','scikit-learn':'1.6.1','scipy':'1.16.3'}
missing=[]
for package,version in required.items():
    try: actual=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: actual=None
    if actual!=version: missing.append(f'{package}=={version}')
if missing:
    print('INSTALANDO:',missing,flush=True)
    p=subprocess.Popen([sys.executable,'-m','pip','install',*missing])
    while p.poll() is None:
        print('DEPENDENCIAS: instalación en curso...',flush=True);time.sleep(10)
    if p.returncode: raise RuntimeError('Falló pip; revisar el log antes de entrenar.')
print('DEPENDENCIAS LISTAS. El motor verifica también PyTorch y torchvision.',flush=True)


In [ ]:
from pathlib import Path
import json

SIGNATURE = "773af3d89fa82d512df8da024e151be6df88ed9a06bb4ff40797612038c6824c"

def choose_resume(input_root, explicit="", output_root=None):
    def inspect(root):
        try:
            state = json.loads((root / "project_state.json").read_text())
        except (OSError, ValueError):
            return None
        if state.get("signature") != SIGNATURE:
            return None
        return len(list((root / "runs").glob("*/complete.json")))

    if explicit:
        source = Path(explicit)
        count = inspect(source)
        if count is None or count < 200:
            raise ValueError("RESUME_DIR debe contener este experimento v5 con al menos 200 corridas.")
    else:
        candidates = []
        for p in Path(input_root).rglob("project_state.json"):
            n = inspect(p.parent)
            if n is not None and n >= 200:
                candidates.append((p.parent, n))
        if len(candidates) != 1:
            raise ValueError(f"Se requiere un único estado v5 con al menos 200 corridas; encontrados {len(candidates)}. Conecta el ZIP más reciente extraído como dataset, o define RESUME_DIR con la carpeta que contiene project_state.json.")
        source, count = candidates[0]
    if output_root is not None:
        existing = Path(output_root)
        if existing.exists() and any(existing.iterdir()):
            output_count = inspect(existing)
            if output_count is None or output_count < count:
                raise ValueError("La carpeta de trabajo contiene un estado anterior o incompatible. Inicia una sesión nueva para continuar desde el dataset seleccionado.")
    print(f"ESTADO SELECCIONADO: {source}", flush=True)
    print(f"CONTINUACION: {count}/240 marcadores previos; se verificará cada predicción antes de entrenar.", flush=True)
    return str(source)

SESSION_HOURS = 8.0  # Pausa a las 7 h 45 min del motor, con 15 min de reserva.
RESUME_DIR = ""  # Detección automática, sin depender del nombre de tu cuenta.
RESUME_DIR = choose_resume("/kaggle/input", RESUME_DIR, "/kaggle/working/RESULTS_closure")


## Motor y validación
La celda siguiente escribe el motor incluido en este notebook. El entrenamiento conserva la implementación de v4.2; el resumen utiliza etiquetas propias de cada conjunto y Wilcoxon exacto por enumeración, con diferencias redondeadas a 12 decimales. FDR de 24 pruebas solo cuando termina la familia completa.

In [ ]:
from pathlib import Path
Path("/kaggle/working/engine_v5.py").write_text('"""Motor reproducible para Kaggle. El notebook incorpora este archivo completo."""\nfrom __future__ import annotations\nimport os, sys, json, time, math, random, hashlib, shutil, threading, traceback, zipfile, gc\nfrom pathlib import Path\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nimport torchvision.transforms as T\nimport timm, imagehash\nfrom sklearn.cluster import AgglomerativeClustering\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support\nfrom scipy.stats import wilcoxon\n\nVERSION = "4.2.0"\nCLASSES = ["healthy", "early_blight", "late_blight"]\nMEAN, STD = (.485, .456, .406), (.229, .224, .225)\nDEFAULT = dict(\n    input_root="/kaggle/input", output_root="/kaggle/working/RESULTS_v4",\n    resume_dir="", session_hours=8.0, reserve_minutes=15,\n    models=["resnet50", "mobilenetv3_large_100", "tf_efficientnetv2_s", "vit_base_patch16_224"],\n    seeds=[42,0,1,7,123,77,88,99,101,2024,11,19,23,31,47],\n    epochs=10, batch_size=32, embedding_batch_size=16, image_size=224,\n    lr=3e-4, weight_decay=1e-4, tau=.10, taus=[.10,.15,.20],\n    mixed_precision=True, require_gpu=True, workers=0, pretrained=True,\n    heartbeat_seconds=60, batch_log_seconds=60, bootstrap_samples=5000,\n    group_candidates=256, split_tolerance=.02, class_split_tolerance=.03, split_solver_seconds=30, preflight=True,\n    # Especificación de fuentes, nombres de carpeta exactos (sin adivinar por ruta completa).\n    # Las dos equivalencias Fungi/Phytopthora son las del protocolo previo: verificar su semántica.\n    sources=[\n        dict(slug="arjuntejaswi/plant-village", domain="potato_lab",\n             labels={"potato___healthy":"healthy","potato___early_blight":"early_blight",\n                     "potato___late_blight":"late_blight"}),\n        dict(slug="warcoder/potato-leaf-disease-dataset", domain="potato_field",\n             labels={"healthy":"healthy","fungi":"early_blight","phytopthora":"late_blight",\n                     "early_blight":"early_blight","late_blight":"late_blight"}),\n        dict(slug="nirmalsankalana/potato-leaf-healthy-and-late-blight", domain="potato_field",\n             labels={"healthy":"healthy"})],\n    # QA local exclusivamente: nunca usar para resultados del artículo.\n    smoke_test=False, fixture_manifest="", domain_filter=[],\n)\ndef canonical(obj):\n    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",",":"), default=str)\ndef digest(obj):\n    return hashlib.sha256(canonical(obj).encode()).hexdigest()\ndef file_hash(p):\n    h=hashlib.sha256()\n    with open(p,"rb") as f:\n        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)\n    return h.hexdigest()\ndef atomic_json(p, obj):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)\n    t=p.with_suffix(p.suffix+".tmp"); t.write_text(canonical(obj),encoding="utf-8"); os.replace(t,p)\ndef atomic_csv(p, df):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)\n    t=p.with_suffix(p.suffix+".tmp"); df.to_csv(t,index=False); os.replace(t,p)\ndef atomic_torch(p, obj):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)\n    t=p.with_suffix(".tmp"); torch.save(obj,t); os.replace(t,p)\ndef seed_all(s):\n    random.seed(s); np.random.seed(s); torch.manual_seed(s)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cudnn.benchmark=False\n\nclass PauseRun(Exception): pass\nclass Monitor:\n    def __init__(self, root, seconds=60):\n        self.root=Path(root); self.root.mkdir(parents=True,exist_ok=True)\n        self.started=time.monotonic(); self.last=self.started; self.phase="inicio"\n        self.lock=threading.RLock(); self.event=threading.Event(); self.seconds=seconds\n    def log(self,msg,advance=True):\n        with self.lock:\n            now=time.monotonic()\n            if advance: self.phase=msg; self.last=now\n            line=f"[{datetime.now(timezone.utc).isoformat(timespec=\'seconds\')}] +{(now-self.started)/60:.1f} min | {msg}"\n            print(line,flush=True)\n            with open(self.root/"execution.log","a",encoding="utf-8") as f: f.write(line+"\\n")\n            atomic_json(self.root/"status.json",dict(elapsed_seconds=now-self.started,\n                last_progress_seconds_ago=now-self.last, phase=self.phase, message=msg))\n    def beat(self):\n        while not self.event.wait(self.seconds):\n            age=time.monotonic()-self.last\n            self.log(f"LATIDO: proceso responde; última operación: {self.phase}; sin hito nuevo hace {age:.0f} s. No certifica avance.",False)\n    def start(self):\n        self.thread=threading.Thread(target=self.beat,daemon=True); self.thread.start()\n    def close(self):\n        self.event.set(); self.thread.join(timeout=2)\n\ndef normalize(s):\n    return s.lower().replace(" ","_").replace("-","_")\ndef scan_manifest(cfg,log):\n    if cfg["fixture_manifest"]:\n        if not cfg["smoke_test"]: raise ValueError("fixture_manifest requiere smoke_test.")\n        df=pd.read_csv(cfg["fixture_manifest"])\n        return df.sort_values("image_id").reset_index(drop=True)\n    root=Path(cfg["input_root"])\n    if not root.exists(): raise FileNotFoundError(f"No existe {root}; adjunta datasets.")\n    rows=[]; skipped=[]; found_sources=0\n    for spec in cfg["sources"]:\n        slug=spec["slug"]; base=slug.split("/")[-1]\n        roots=sorted(p for p in root.rglob(base) if p.is_dir())\n        if len(roots)>1:\n            raise ValueError(f"Varias versiones de {slug}: {roots}. Adjunta solo una.")\n        if not roots:\n            log.log(f"Fuente no adjunta: {slug}"); continue\n        found_sources+=1; folder=roots[0]\n        images=sorted(p for p in folder.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"} and p.is_file())\n        log.log(f"MANIFIESTO {slug}: {len(images)} archivos; leyendo etiquetas y SHA-256.")\n        for i,p in enumerate(images):\n            rel=p.relative_to(folder).as_posix()\n            labels={spec["labels"][normalize(x)] for x in p.relative_to(folder).parts[:-1] if normalize(x) in spec["labels"]}\n            if not labels: skipped.append(dict(source=slug,relative_path=rel)); continue\n            if len(labels)!=1: raise ValueError(f"Etiquetas ambiguas: {p}")\n            label=labels.pop()\n            with Image.open(p) as im: im.verify()\n            h=file_hash(p)\n            rows.append(dict(image_id=digest([slug,rel])[:24],path=str(p),relative_path=rel,\n                source=slug,domain=spec["domain"],label=label,label_id=CLASSES.index(label),sha256=h))\n            if (i+1)%500==0: log.log(f"MANIFIESTO {slug}: {i+1}/{len(images)} leídos.")\n    if not rows: raise ValueError("Sin imágenes de las fuentes configuradas. Revisa sources; no se aceptan etiquetas inferidas del nombre del dataset.")\n    df=pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)\n    if cfg["domain_filter"]: df=df[df.domain.isin(cfg["domain_filter"])].reset_index(drop=True)\n    if df.empty: raise ValueError("domain_filter dejó el manifiesto vacío.")\n    if df.image_id.duplicated().any(): raise ValueError("IDs repetidos.")\n    conflict=df.groupby("sha256").label.nunique()\n    if (conflict>1).any(): raise ValueError("Mismo archivo con etiquetas distintas; corregir datos antes de entrenar.")\n    atomic_csv(Path(cfg["output_root"])/"skipped_images.csv",pd.DataFrame(skipped,columns=["source","relative_path"]))\n    for name,g in df.groupby("domain"):\n        cnt=g.label.value_counts()\n        if set(cnt.index)!=set(CLASSES) or cnt.min()<7:\n            raise ValueError(f"{name}: se necesitan las tres clases y >=7 imágenes/clase; encontradas {cnt.to_dict()}.")\n    log.log(f"MANIFIESTO: {len(df)} imágenes; {found_sources} fuentes; {df.sha256.duplicated(False).sum()} archivos en grupos exactos repetidos.")\n    print(df.groupby(["domain","source","label"]).size().to_string(),flush=True)\n    return df\n\nclass Images(Dataset):\n    def __init__(self,df,tf): self.paths=df.path.tolist(); self.y=df.label_id.to_numpy(); self.tf=tf\n    def __len__(self): return len(self.paths)\n    def __getitem__(self,i):\n        with Image.open(self.paths[i]) as im: x=self.tf(im.convert("RGB"))\n        return x,int(self.y[i])\ndef loader(df,tf,cfg,shuffle=False,seed=0):\n    return DataLoader(Images(df,tf),batch_size=cfg["batch_size"],shuffle=shuffle,\n        num_workers=cfg["workers"],pin_memory=torch.cuda.is_available(),\n        generator=torch.Generator().manual_seed(seed),drop_last=False)\ndef cpu_state(model):\n    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\ndef build_model(name,cfg,device,pretrained=None):\n    if cfg["smoke_test"]:\n        return nn.Sequential(nn.Conv2d(3,6,3,padding=1),nn.ReLU(),\n            nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(6,3)).to(device)\n    return timm.create_model(name,pretrained=cfg["pretrained"] if pretrained is None else pretrained,\n        num_classes=3).to(device)\ndef transforms(cfg,train=False):\n    ops=[T.Resize((cfg["image_size"],cfg["image_size"]))]\n    if train: ops += [T.RandomHorizontalFlip(),T.RandomRotation(10)]\n    return T.Compose(ops+[T.ToTensor(),T.Normalize(MEAN,STD)])\n\ndef prepare_embeddings(df,directory,cfg,device,log):\n    directory.mkdir(parents=True,exist_ok=True); cache=directory/"embeddings.npz"\n    if cache.exists():\n        with np.load(cache,allow_pickle=False) as a:\n            if list(a["image_id"])!=df.image_id.tolist(): raise ValueError("Cache de embeddings incompatible.")\n            z=a["z"].copy(); ph=a["phash"].copy()\n        log.log(f"EMBEDDINGS {df.domain.iloc[0]}: caché válido, omitido."); return z,ph\n    log.log(f"EMBEDDINGS {df.domain.iloc[0]}: {\'QA sintético\' if cfg[\'smoke_test\'] else \'cargando DINOv2 (puede descargar pesos)\'}")\n    if cfg["smoke_test"]:\n        rng=np.random.default_rng(31); z=rng.normal(size=(len(df),12))\n        z/=np.linalg.norm(z,axis=1,keepdims=True)\n    else:\n        dino=timm.create_model("vit_small_patch14_dinov2.lvd142m",pretrained=True,num_classes=0).to(device).eval()\n        tf=T.Compose([T.Resize((518,518)),T.ToTensor(),T.Normalize(MEAN,STD)])\n        dl=DataLoader(Images(df,tf),batch_size=cfg["embedding_batch_size"],num_workers=cfg["workers"])\n        chunks=[]; last=time.monotonic()\n        with torch.no_grad():\n            for b,(x,_) in enumerate(dl,1):\n                chunks.append(dino(x.to(device)).float().cpu().numpy())\n                if b==1 or b==len(dl) or time.monotonic()-last>=cfg["batch_log_seconds"]:\n                    log.log(f"EMBEDDINGS lote {b}/{len(dl)}"); last=time.monotonic()\n        z=np.concatenate(chunks); z/=np.linalg.norm(z,axis=1,keepdims=True).clip(min=1e-12)\n        del dino,chunks; gc.collect()\n        if device.type=="cuda": torch.cuda.empty_cache()\n    ph=[]\n    for i,p in enumerate(df.path):\n        with Image.open(p) as im: ph.append(imagehash.phash(im.convert("RGB")).hash.flatten())\n        if (i+1)%1000==0: log.log(f"pHash {i+1}/{len(df)}")\n    ph=np.asarray(ph,dtype=np.uint8)\n    tmp=cache.with_suffix(".tmp.npz")\n    np.savez_compressed(tmp,image_id=df.image_id.to_numpy(dtype=str),z=z,phash=ph)\n    os.replace(tmp,cache); log.log(f"EMBEDDINGS y pHash guardados: {len(z)} imágenes.")\n    return z,ph\n\n"""Partición 70/15/15 por imágenes con grupos indivisibles y diagnóstico explícito."""\nimport numpy as np\nfrom scipy.optimize import milp, Bounds, LinearConstraint\nfrom scipy.sparse import lil_matrix\n\nclass SplitInfeasible(ValueError):\n    """El solver demostró que las restricciones no admiten una partición."""\nclass SplitSearchLimit(RuntimeError):\n    """La búsqueda terminó sin encontrar solución; no demuestra inviabilidad."""\n\ndef grouped_split(y,groups,seed,cfg):\n    y=np.asarray(y,dtype=int); groups=np.asarray(groups)\n    if len(y)!=len(groups) or set(y)!={0,1,2}:\n        raise ValueError("Se necesitan etiquetas 0,1,2 y un grupo por imagen.")\n    unique,inverse=np.unique(groups,return_inverse=True); n_groups=len(unique)\n    if n_groups<3: raise SplitInfeasible("Menos de tres grupos indivisibles.")\n    counts=np.zeros((n_groups,3),dtype=int)\n    np.add.at(counts,(inverse,y),1)\n    sizes=counts.sum(1); n=len(y); target=np.array([.70,.15,.15])\n    tolerance=float(cfg["split_tolerance"])\n    lo=np.ceil((target-tolerance).clip(0)*n-1e-9)\n    hi=np.floor((target+tolerance).clip(max=1)*n+1e-9)\n    if sizes.max()>hi.max():\n        raise SplitInfeasible(f"Un grupo tiene {sizes.max()}/{n} imágenes; supera el máximo admisible {int(hi.max())}.")\n    for c in range(3):\n        if np.count_nonzero(counts[:,c])<3:\n            raise SplitInfeasible(f"La clase {c} aparece en menos de tres grupos indivisibles.")\n\n    class_tolerance=float(cfg.get("class_split_tolerance",.03))\n    class_totals=counts.sum(0)\n    class_lo=np.maximum(1,np.ceil((target[:,None]-class_tolerance).clip(0)*class_totals-1e-9))\n    class_hi=np.floor((target[:,None]+class_tolerance).clip(max=1)*class_totals+1e-9)\n    for c in range(3):\n        if counts[:,c].max()>class_hi[:,c].max():\n            raise SplitInfeasible(f"Clase {c}: un grupo tiene {counts[:,c].max()}/{class_totals[c]} imágenes, superior al máximo por clase admisible {int(class_hi[:,c].max())}.")\n    # x[g,s] binary; d[s,k] nonnegative absolute deviation for total and each class.\n    features=np.column_stack([sizes,counts]); totals=features.sum(0)\n    nx=3*n_groups; nv=nx+12\n    matrix=lil_matrix((n_groups+12+24,nv),dtype=float)\n    lower=np.full(matrix.shape[0],-np.inf); upper=np.full(matrix.shape[0],np.inf)\n    row=0\n    for g in range(n_groups):\n        matrix[row,3*g:3*g+3]=1; lower[row]=upper[row]=1; row+=1\n    for s in range(3):\n        indices=np.arange(n_groups)*3+s\n        for k in range(4):\n            matrix[row,indices]=features[:,k]\n            lower[row]=lo[s] if k==0 else class_lo[s,k-1]\n            upper[row]=hi[s] if k==0 else class_hi[s,k-1]\n            row+=1\n    for s in range(3):\n        indices=np.arange(n_groups)*3+s\n        for k in range(4):\n            wanted=target[s]*totals[k]; dev=nx+4*s+k\n            matrix[row,indices]=features[:,k]; matrix[row,dev]=-1\n            upper[row]=wanted; row+=1\n            matrix[row,indices]=-features[:,k]; matrix[row,dev]=-1\n            upper[row]=-wanted; row+=1\n    objective=np.zeros(nv)\n    objective[nx:]=np.tile(1/np.maximum(totals,1),3)\n    # Seed affects tie-breaking, never feasibility, labels or target proportions.\n    objective[:nx]=np.random.default_rng(seed).uniform(0,1e-7/n_groups,nx)\n    integrality=np.zeros(nv); integrality[:nx]=1\n    bound_hi=np.full(nv,np.inf); bound_hi[:nx]=1\n    solved=milp(c=objective,integrality=integrality,\n        bounds=Bounds(np.zeros(nv),bound_hi),\n        constraints=LinearConstraint(matrix.tocsc(),lower,upper),\n        options=dict(time_limit=float(cfg.get("split_solver_seconds",30)),mip_rel_gap=.01))\n    if solved.status==2:\n        raise SplitInfeasible(\n            f"No existe partición 70/15/15 con tolerancia ±{100*tolerance:.0f} puntos, "\n            f"grupos intactos y balance por clase. n={n}, grupos={n_groups}, "\n            f"mayor grupo={sizes.max()}. No cambiar tau según exactitud del modelo.")\n    if solved.x is None:\n        raise SplitSearchLimit(f"Búsqueda sin solución verificada (status={solved.status}): {solved.message}")\n    x=solved.x[:nx].reshape(n_groups,3)\n    if not np.allclose(x,np.rint(x),atol=1e-5):\n        raise SplitSearchLimit("El solver no devolvió asignaciones enteras verificables.")\n    assignment=x.argmax(1)\n    parts=tuple(np.flatnonzero(assignment[inverse]==s) for s in range(3))\n    for s,p in enumerate(parts):\n        if not lo[s]<=len(p)<=hi[s] or set(y[p])!={0,1,2}:\n            raise SplitSearchLimit("La solución no superó la validación independiente.")\n        observed=np.bincount(y[p],minlength=3)\n        if np.any(observed<class_lo[s]) or np.any(observed>class_hi[s]):\n            raise SplitSearchLimit("La solución no respeta el balance por clase.")\n    return parts\n\n\ndef random_split(y,seed):\n    ix=np.arange(len(y)); tr,r=train_test_split(ix,test_size=.3,stratify=y,random_state=seed)\n    va,te=train_test_split(r,test_size=.5,stratify=np.asarray(y)[r],random_state=seed)\n    return tuple(np.sort(k) for k in (tr,va,te))\ndef verify_split(parts,y,groups=None):\n    tr,va,te=parts\n    assert len(set(tr)&set(va))==len(set(tr)&set(te))==len(set(va)&set(te))==0\n    assert sorted(np.concatenate(parts).tolist())==list(range(len(y)))\n    for k in parts:\n        if set(np.asarray(y)[k])!={0,1,2}: raise ValueError("Partición sin las tres clases.")\n    if groups is not None:\n        gs=[set(np.asarray(groups)[k]) for k in parts]\n        assert not (gs[0]&gs[1] or gs[0]&gs[2] or gs[1]&gs[2])\n\ndef prepare_groups_splits(df,z,ph,directory,cfg,log):\n    saved=directory/"clusters.csv"\n    if saved.exists():\n        g=pd.read_csv(saved)\n        if g.image_id.tolist()!=df.image_id.tolist(): raise ValueError("Clusters incompatibles.")\n        clusters={t:g[f"tau_{t}"].to_numpy() for t in cfg["taus"]}\n        log.log("CLUSTERS: caché válido.")\n    else:\n        clusters={}\n        for tau in cfg["taus"]:\n            log.log(f"CLUSTERING average/cosine: tau={tau}, n={len(z)}. Etapa en CPU.")\n            started=time.monotonic()\n            clusters[tau]=AgglomerativeClustering(n_clusters=None,metric="cosine",linkage="average",distance_threshold=tau).fit_predict(z)\n            log.log(f"CLUSTERING tau={tau}: {len(np.unique(clusters[tau]))} grupos en {time.monotonic()-started:.1f} s.")\n        atomic_csv(saved,pd.DataFrame(dict(image_id=df.image_id,**{f"tau_{t}":g for t,g in clusters.items()})))\n    diag=[]\n    for t,g in clusters.items():\n        sizes=np.bincount(g)\n        diag.append(dict(tau=t,clusters=len(sizes),images=len(g),max_size=int(sizes.max()),\n            singleton_image_percent=100*float(np.mean(sizes[g]==1))))\n    atomic_csv(directory/"cluster_diagnostics.csv",pd.DataFrame(diag))\n    size_rows=[]\n    for t,g in clusters.items():\n        sizes=np.bincount(g)\n        for group in np.unique(g):\n            mask=g==group\n            cc=np.bincount(df.label_id.to_numpy()[mask],minlength=3)\n            size_rows.append(dict(tau=t,group=int(group),images=int(mask.sum()),\n                healthy=int(cc[0]),early_blight=int(cc[1]),late_blight=int(cc[2])))\n        log.log(f"DIAGNOSTICO tau={t}: mayor grupo={sizes.max()}/{len(g)} ({100*sizes.max()/len(g):.1f}%).")\n    atomic_csv(directory/"cluster_sizes_by_class.csv",pd.DataFrame(size_rows))\n\n    plans={}\n    for seed in cfg["seeds"]:\n        for strategy in ["random","group"]:\n            p=directory/f"split_{strategy}_{seed}.csv"\n            if p.exists():\n                saved_split=pd.read_csv(p)\n                if saved_split.image_id.tolist()!=df.image_id.tolist(): raise ValueError("Split incompatible.")\n                parts=tuple(np.flatnonzero(saved_split["split"].eq(s)) for s in ["train","val","test"])\n            else:\n                log.log(f"BUSCANDO SPLIT {df.domain.iloc[0]} {strategy} seed={seed}; límite solver={cfg[\'split_solver_seconds\']} s.")\n                try:\n                    parts=random_split(df.label_id.to_numpy(),seed) if strategy=="random" else grouped_split(df.label_id.to_numpy(),clusters[cfg["tau"]],seed,cfg)\n                except (SplitInfeasible,SplitSearchLimit) as exc:\n                    atomic_json(directory/"split_failure.json",dict(error=type(exc).__name__,\n                        message=str(exc),tau=cfg["tau"],seed=seed,strategy=strategy))\n                    raise\n                labels=np.full(len(df),"",dtype=object)\n                for k,n in zip(parts,["train","val","test"]): labels[k]=n\n                saved_split=df[["image_id","source","label","label_id","sha256"]].copy()\n                saved_split["split"]=labels; saved_split["visual_group"]=clusters[cfg["tau"]]\n                atomic_csv(p,saved_split)\n            verify_split(parts,df.label_id.to_numpy(),clusters[cfg["tau"]] if strategy=="group" else None)\n            target=np.array([.70,.15,.15])\n            counts=np.bincount(df.label_id.to_numpy(),minlength=3)\n            for part_index,indices in enumerate(parts):\n                if abs(len(indices)/len(df)-target[part_index])>cfg["split_tolerance"]+1e-9:\n                    raise ValueError("Split cacheado fuera de tolerancia global.")\n                observed=np.bincount(df.label_id.to_numpy()[indices],minlength=3)\n                if np.max(np.abs(observed/counts-target[part_index]))>cfg["class_split_tolerance"]+1e-9:\n                    raise ValueError("Split cacheado fuera de tolerancia por clase.")\n            plans[(strategy,seed)]=parts\n            log.log(f"SPLIT {df.domain.iloc[0]} {strategy} semilla={seed}: train/val/test={list(map(len,parts))}; validado.")\n    # Exact exposure is measurable without pretending visual clusters are verified duplicates.\n    exposure=[]\n    for (strategy,seed),(tr,va,te) in plans.items():\n        exposed=df.sha256.iloc[te].isin(set(df.sha256.iloc[tr]))\n        exposure.append(dict(strategy=strategy,seed=seed,test_images=len(te),\n            exact_exposed_images=int(exposed.sum()),exact_exposure_rate=float(exposed.mean()),\n            near_duplicate_exposure="NOT_MEASURED"))\n    atomic_csv(directory/"exact_duplicate_exposure.csv",pd.DataFrame(exposure))\n    # Uniform pair sample: descriptive manual audit only, no unsupported population recall/purity.\n    rng=np.random.default_rng(42); pairs=set(); target=min(500,len(df)*(len(df)-1)//2)\n    while len(pairs)<target:\n        a,b=sorted(rng.choice(len(df),2,replace=False).tolist()); pairs.add((a,b))\n    annotations=[]\n    for a,b in sorted(pairs):\n        annotations.append(dict(image_a=df.image_id.iloc[a],image_b=df.image_id.iloc[b],\n            path_a=df.path.iloc[a],path_b=df.path.iloc[b],\n            cosine_distance=float(np.clip(1-np.dot(z[a],z[b]),0,2)),\n            phash_hamming=int(np.count_nonzero(ph[a]!=ph[b])),\n            same_visual_cluster=int(clusters[cfg["tau"]][a]==clusters[cfg["tau"]][b]),\n            same_sha256=int(df.sha256.iloc[a]==df.sha256.iloc[b]),manual_duplicate=""))\n    atomic_csv(directory/"manual_pair_sample.csv",pd.DataFrame(annotations))\n    return plans\n\ndef metric_values(y,p):\n    return dict(accuracy=float(accuracy_score(y,p)),balanced_accuracy=float(balanced_accuracy_score(y,p)),\n        macro_f1=float(f1_score(y,p,labels=[0,1,2],average="macro",zero_division=0)))\ndef finished(run_dir,signature,expected_ids=None):\n    marker=run_dir/"complete.json"; pred=run_dir/"predictions.csv"\n    if not marker.exists(): return False\n    info=json.loads(marker.read_text())\n    if info["signature"]!=signature or not pred.exists() or file_hash(pred)!=info["predictions_sha256"]:\n        raise ValueError(f"Resultados corruptos o incompatibles: {run_dir}")\n    if expected_ids is not None:\n        got=pd.read_csv(pred)\n        if got.image_id.tolist()!=list(expected_ids): raise ValueError("IDs de predicción no coinciden con test.")\n    return True\n\ndef train_one(df,parts,domain,name,strategy,seed,run_dir,cfg,signature,device,log,deadline):\n    tr,va,te=parts; run_dir.mkdir(parents=True,exist_ok=True)\n    if finished(run_dir,signature,df.image_id.iloc[te]):\n        log.log(f"OMITIDO: {domain}/{name}/{strategy}/{seed} ya terminado y verificado."); return True\n    if time.monotonic()>=deadline: raise PauseRun("Presupuesto de sesión alcanzado antes de nueva corrida.")\n    log.log(f"INICIO MODELO {domain}/{name}/{strategy}, semilla={seed}.")\n    ckpt=run_dir/"last.pt"; state=None\n    if ckpt.exists():\n        log.log("REANUDAR: leyendo checkpoint seguro (modelo, optimizador, época y mejor modelo).")\n        state=torch.load(ckpt,map_location="cpu",weights_only=True)\n        if state["signature"]!=signature: raise ValueError("Checkpoint incompatible.")\n    seed_all(seed)\n    m=build_model(name,cfg,device,pretrained=False if state else None)\n    opt=torch.optim.AdamW(m.parameters(),lr=cfg["lr"],weight_decay=cfg["weight_decay"])\n    amp=cfg["mixed_precision"] and device.type=="cuda"\n    scaler=torch.amp.GradScaler("cuda",enabled=amp)\n    start=0; best=-1.; best_state=None; history=[]\n    if state:\n        m.load_state_dict(state["model"]); opt.load_state_dict(state["optimizer"])\n        scaler.load_state_dict(state["scaler"]); start=state["epoch"]; best=state["best"]\n        best_state=state["best_model"]; history=state["history"]\n        del state\n        log.log(f"REANUDAR desde época {start+1}/{cfg[\'epochs\']}; {start} épocas guardadas.")\n    train_dl=loader(df.iloc[tr],transforms(cfg,True),cfg,True,seed)\n    val_dl=loader(df.iloc[va],transforms(cfg),cfg)\n    try:\n        for epoch in range(start,cfg["epochs"]):\n            if time.monotonic()>=deadline: raise PauseRun("Pausa antes de nueva época; checkpoint preservado.")\n            epoch_seed=(seed*100003+epoch)%(2**32-1)\n            seed_all(epoch_seed); train_dl.generator.manual_seed(epoch_seed)\n            begin=time.monotonic(); log.log(f"EPOCA {epoch+1}/{cfg[\'epochs\']} | {name} {strategy} seed={seed}: {len(train_dl)} lotes.")\n            m.train(); loss_sum=0.; count=0; last=time.monotonic()\n            for b,(x,y) in enumerate(train_dl,1):\n                x=x.to(device); y=y.to(device); opt.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type,enabled=amp):\n                    loss=nn.functional.cross_entropy(m(x),y)\n                if not torch.isfinite(loss): raise FloatingPointError("Pérdida NaN/Inf.")\n                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()\n                loss_sum+=float(loss.detach())*len(y); count+=len(y)\n                if b==1 or b==len(train_dl) or time.monotonic()-last>=cfg["batch_log_seconds"]:\n                    elapsed=time.monotonic()-begin\n                    log.log(f"LOTE {b}/{len(train_dl)}, época {epoch+1}, loss={loss_sum/count:.5f}; ETA entrenamiento época={(elapsed/b*(len(train_dl)-b))/60:.1f} min.")\n                    last=time.monotonic()\n                if time.monotonic()>=deadline:\n                    raise PauseRun("Límite de sesión: se conserva la última época completa; la época parcial se repetirá.")\n            m.eval(); yy=[]; pp=[]\n            log.log(f"VALIDACION época {epoch+1}: {len(val_dl)} lotes.")\n            with torch.no_grad():\n                for x,y in val_dl:\n                    with torch.autocast(device_type=device.type,enabled=amp): o=m(x.to(device))\n                    yy.extend(y.tolist()); pp.extend(o.argmax(1).cpu().tolist())\n            score=metric_values(yy,pp)["macro_f1"]\n            if score>best: best=score; best_state=cpu_state(m)\n            history.append(dict(epoch=epoch+1,loss=loss_sum/count,val_macro_f1=score,seconds=time.monotonic()-begin))\n            log.log(f"GUARDANDO checkpoint época {epoch+1}, val_macro_f1={score:.5f}, mejor={best:.5f}.")\n            atomic_torch(ckpt,dict(signature=signature,epoch=epoch+1,model=cpu_state(m),\n                optimizer=opt.state_dict(),scaler=scaler.state_dict(),best=best,best_model=best_state,history=history))\n            atomic_csv(run_dir/"history.csv",pd.DataFrame(history))\n            log.log(f"EPOCA {epoch+1} GUARDADA en {time.monotonic()-begin:.1f} s; {cfg[\'epochs\']-epoch-1} épocas restantes en esta corrida.")\n        if best_state is None: raise RuntimeError("No hay estado válido para test.")\n        log.log(f"TEST {name}/{strategy}/{seed}: {len(te)} imágenes; carga mejor época por validación.")\n        m.load_state_dict(best_state); m.eval(); probabilities=[]; yy=[]\n        with torch.no_grad():\n            for x,y in loader(df.iloc[te],transforms(cfg),cfg):\n                with torch.autocast(device_type=device.type,enabled=amp): o=m(x.to(device))\n                probabilities.append(o.float().softmax(1).cpu().numpy()); yy.extend(y.tolist())\n        probs=np.concatenate(probabilities); preds=probs.argmax(1)\n        out=pd.DataFrame(dict(image_id=df.image_id.iloc[te].to_numpy(),y_true=yy,y_pred=preds))\n        for i,c in enumerate(CLASSES): out[f"prob_{c}"]=probs[:,i]\n        atomic_csv(run_dir/"predictions.csv",out)\n        atomic_json(run_dir/"complete.json",dict(signature=signature,domain=domain,model=name,\n            strategy=strategy,seed=seed,metrics=metric_values(yy,preds),epochs=cfg["epochs"],\n            predictions_sha256=file_hash(run_dir/"predictions.csv")))\n        # Only last.pt in this exact completed run is obsolete; retain small outputs/history.\n        ckpt.unlink(missing_ok=True)\n        log.log(f"CORRIDA COMPLETA {domain}/{name}/{strategy}/{seed}; predicciones guardadas.")\n        return True\n    finally:\n        del m,opt,scaler,best_state,train_dl,val_dl\n        gc.collect()\n        if device.type=="cuda": torch.cuda.empty_cache()\n\ndef cliffs_delta(a,b):\n    return float(np.sign(np.asarray(a)[:,None]-np.asarray(b)[None,:]).mean())\ndef benjamini_hochberg(p):\n    p=np.asarray(p,float); order=np.argsort(p); adjusted=np.empty(len(p))\n    ranked=p[order]*len(p)/np.arange(1,len(p)+1)\n    adjusted[order]=np.minimum(1,np.minimum.accumulate(ranked[::-1])[::-1])\n    return adjusted\ndef summarize(root,cfg,signature,log):\n    domain_labels=json.loads((root/"project_state.json").read_text())["scientific"]["domain_labels"]\n    rows=[]; pc=[]\n    for f in sorted((root/"runs").glob("*/complete.json")):\n        info=json.loads(f.read_text())\n        finished(f.parent,signature)\n        key={k:info[k] for k in ["domain","model","strategy","seed"]}\n        rows.append(dict(**key,**info["metrics"]))\n        g=pd.read_csv(f.parent/"predictions.csv")\n        pr,re,f1,su=precision_recall_fscore_support(g.y_true,g.y_pred,labels=[0,1,2],zero_division=0)\n        for i,c in enumerate(domain_labels[info["domain"]]): pc.append(dict(**key,label=c,precision=pr[i],recall=re[i],f1=f1[i],support=int(su[i])))\n    metrics=pd.DataFrame(rows,columns=["domain","model","strategy","seed","accuracy","balanced_accuracy","macro_f1"])\n    atomic_csv(root/"metrics_by_run.csv",metrics)\n    atomic_csv(root/"per_class_metrics.csv",pd.DataFrame(pc))\n    stats=[]\n    for (domain,model),g in metrics.groupby(["domain","model"]):\n        for metric in ["accuracy","balanced_accuracy","macro_f1"]:\n            pivot=g.pivot(index="seed",columns="strategy",values=metric)\n            if not {"random","group"}.issubset(pivot.columns): continue\n            pivot=pivot.dropna(); a=pivot.random.to_numpy(); b=pivot["group"].to_numpy(); n=len(a)\n            if n==0: continue\n            d=a-b; lo=hi=llo=lhi=float("nan"); p=float("nan")\n            if n>=2:\n                rng=np.random.default_rng(123); inds=rng.integers(0,n,size=(cfg["bootstrap_samples"],n))\n                am=a[inds].mean(1); bm=b[inds].mean(1)\n                lo,hi=np.percentile(am-bm,[2.5,97.5])\n                if np.all(am>0): llo,lhi=np.percentile(100*(am-bm)/am,[2.5,97.5])\n                p=1. if np.all(d==0) else exact_wilcoxon(d)\n            stats.append(dict(domain=domain,model=model,metric=metric,n_pairs=n,\n                expected_pairs=len(cfg["seeds"]),complete=n==len(cfg["seeds"]),\n                random_mean=float(a.mean()),group_mean=float(b.mean()),difference=float(d.mean()),\n                difference_ci95_low=lo,difference_ci95_high=hi,\n                lsi_percent=float(100*d.mean()/a.mean()) if a.mean()>0 else float("nan"),\n                lsi_ci95_low=llo,lsi_ci95_high=lhi,cliffs_delta=cliffs_delta(a,b),wilcoxon_p=p))\n    st=pd.DataFrame(stats)\n    if not st.empty:\n        st["p_fdr_bh"]=np.nan\n        # FDR is released only when the prespecified family is complete.\n        domains=sorted(pd.read_csv(root/"manifest.csv").domain.unique())\n        expected=len(domains)*len(cfg["models"])*3\n        if len(st)==expected and st.complete.all() and st.wilcoxon_p.notna().all():\n            st["p_fdr_bh"]=benjamini_hochberg(st.wilcoxon_p)\n    atomic_csv(root/"stats.csv",st)\n    log.log(f"RESUMEN: {len(metrics)} corridas terminadas. Resultados parciales no equivalen a estudio completo.")\n    return len(metrics)\n\ndef exact_wilcoxon(d):\n    from scipy.stats import rankdata\n    rounded=np.round(d,12); nz=rounded[rounded!=0]\n    if len(nz)==0: return 1.\n    ranks=rankdata(abs(nz),method=\'average\')\n    signs=2*((np.arange(2**len(nz))[:,None] >> np.arange(len(nz))) & 1)-1\n    return float((abs(signs@ranks)>=abs(np.sign(nz)@ranks)-1e-12).mean())\n',encoding="utf-8")
Path("/kaggle/working/runtime_v5.py").write_text('"""Frozen v5 evaluation. Uses the unchanged v4.2 training implementation."""\nimport json, time, shutil, zipfile, traceback, sys, os, gc, importlib.metadata\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nimport engine_v5 as base\n\nSIGNATURE = \'773af3d89fa82d512df8da024e151be6df88ed9a06bb4ff40797612038c6824c\'\n\ndef validate(root, signature=SIGNATURE):\n    state=json.loads((root/\'project_state.json\').read_text())\n    if state[\'version\']!=\'5.0.0\' or state[\'signature\']!=signature:\n        raise ValueError(\'Estado incompatible: se requiere este experimento v5, no RESULTS_v4.\')\n    if base.digest(dict(scientific=state[\'scientific\'],input_files=state[\'input_files\']))!=signature:\n        raise ValueError(\'La especificación del experimento cambió.\')\n    for rel, expected in state[\'input_files\'].items():\n        p=(root/rel).resolve()\n        if not p.is_relative_to(root.resolve()) or base.file_hash(p)!=expected:\n            raise ValueError(f\'Archivo alterado o ausente: {rel}\')\n    df=pd.read_csv(root/\'manifest.csv\'); plans={}; sci=state[\'scientific\']\n    if not df.image_id.is_unique: raise ValueError(\'IDs de imagen repetidos.\')\n    for domain,labels in sci[\'domain_labels\'].items():\n        d=df[df.domain.eq(domain)].reset_index(drop=True)\n        if d.empty or not d.label.equals(d.label_id.map(dict(enumerate(labels)))):\n            raise ValueError(\'Etiquetas incompatibles con el protocolo.\')\n        if domain==\'potato_field_single_source\' and (len(d)!=1296 or d.source.nunique()!=1):\n            raise ValueError(\'El control de fuente de campo no coincide.\')\n        if domain==\'potato_lab_leaf_id\' and (len(d)!=2152 or d.leaf_id.isna().any() or d.leaf_id.nunique()!=538):\n            raise ValueError(\'Referencia de identidad incompleta.\')\n        local={}\n        for seed in sci[\'seeds\']:\n            for strategy in [\'random\',\'group\']:\n                s=pd.read_csv(root/\'data\'/domain/f\'split_{strategy}_{seed}.csv\')\n                if s.image_id.tolist()!=d.image_id.tolist() or s.label_id.tolist()!=d.label_id.tolist():\n                    raise ValueError(\'Orden de IDs o etiquetas de partición alterado.\')\n                parts=tuple(np.flatnonzero(s.split.eq(k)) for k in [\'train\',\'val\',\'test\'])\n                base.verify_split(parts,d.label_id.to_numpy(),d.evaluation_group.to_numpy() if strategy==\'group\' else None)\n                for part,target in zip(parts,[.7,.15,.15]):\n                    counts=np.bincount(d.label_id.to_numpy()[part],minlength=3)\n                    if abs(len(part)/len(d)-target)>.020000001 or max(abs(counts/np.bincount(d.label_id,minlength=3)-target))>.030000001:\n                        raise ValueError(\'Partición fuera de tolerancia.\')\n                local[(strategy,seed)]=parts\n        plans[domain]=(d,local)\n    allowed=set()\n    completed=0\n    for domain,(d,local) in plans.items():\n        for name in sci[\'models\']:\n            for (strategy,seed),parts in local.items():\n                rid=f\'{domain}__{name}__{strategy}__{seed}\';allowed.add(rid)\n                rd=root/\'runs\'/rid\n                if base.finished(rd,signature,d.image_id.iloc[parts[2]]):\n                    info=json.loads((rd/\'complete.json\').read_text()); pred=pd.read_csv(rd/\'predictions.csv\')\n                    if any(info[k]!=v for k,v in dict(domain=domain,model=name,strategy=strategy,seed=seed,epochs=sci[\'epochs\']).items()):\n                        raise ValueError(\'Marcador de corrida incompatible.\')\n                    prob=pred[[f\'prob_{c}\' for c in sci[\'domain_labels\'][domain]]].to_numpy()\n                    if not np.isfinite(prob).all() or (prob<0).any() or not np.allclose(prob.sum(1),1,atol=1e-5) or not np.array_equal(prob.argmax(1),pred.y_pred):\n                        raise ValueError(\'Probabilidades inválidas.\')\n                    if pred.y_true.tolist()!=d.label_id.iloc[parts[2]].tolist(): raise ValueError(\'Verdad de test alterada.\')\n                    for k,v in base.metric_values(pred.y_true,pred.y_pred).items():\n                        if not np.isclose(v,info[\'metrics\'][k],atol=1e-12): raise ValueError(\'Métrica no reproducible.\')\n                    completed+=1\n    if any(p.parent.name not in allowed for p in (root/\'runs\').glob(\'*/complete.json\')):\n        raise ValueError(\'Resultados de otro experimento en la carpeta.\')\n    return state,plans,completed\n\ndef resolve_images(plans,input_root,log):\n    index={}\n    log.log(\'VERIFICANDO imágenes: indexación de archivos; no extrae embeddings ni recalcula particiones.\')\n    for p in input_root.rglob(\'*\'):\n        if p.is_file() and p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\'}:\n            index.setdefault(p.name,[]).append(p)\n    total=sum(len(d) for d,_ in plans.values()); done=0\n    for domain,(d,_) in plans.items():\n        resolved=[]\n        for row in d.itertuples():\n            old=Path(row.path); rel=Path(row.relative_path)\n            candidates=[old] if old.is_file() and old.resolve().is_relative_to(input_root.resolve()) else []\n            candidates+=index.get(rel.name,[])\n            chosen=None\n            for p in dict.fromkeys(candidates):\n                if base.file_hash(p)==row.sha256:\n                    with Image.open(p) as im: im.verify()\n                    chosen=p;break\n            if chosen is None: raise FileNotFoundError(f\'Imagen ausente o bytes distintos: {row.source}/{row.relative_path}\')\n            resolved.append(str(chosen)); done+=1\n            if done%250==0 or done==total: log.log(f\'IMAGENES VERIFICADAS {done}/{total}: SHA-256 y decodificación OK.\')\n        d[\'path\']=resolved\n\ndef export(root,log):\n    target=root.parent/\'results_closure_v5.zip\'; temp=target.with_suffix(\'.tmp.zip\')\n    log.log(\'EXPORTANDO resultados y checkpoint pendiente; conservar ZIP para continuar.\')\n    with zipfile.ZipFile(temp,\'w\',allowZip64=True) as z:\n        for p in sorted(root.rglob(\'*\')):\n            if p.is_file() and \'.tmp\' not in p.name:\n                z.write(p,\'RESULTS_closure/\'+p.relative_to(root).as_posix(),compress_type=zipfile.ZIP_STORED if p.suffix==\'.pt\' else zipfile.ZIP_DEFLATED)\n    os.replace(temp,target)\n    log.log(f\'ZIP LISTO: {target}, {target.stat().st_size/1e6:.1f} MB.\')\n\ndef initialize(root, input_root, resume_dir=\'\'):\n    if (root/\'project_state.json\').exists(): return\n    if root.exists() and any(root.iterdir()): raise ValueError(\'Carpeta de salida no vacía sin estado válido.\')\n    if resume_dir:\n        source=Path(resume_dir)\n    else:\n        candidates=[]\n        for p in input_root.rglob(\'project_state.json\'):\n            try:\n                if json.loads(p.read_text()).get(\'signature\')==SIGNATURE: candidates.append(p.parent)\n            except (ValueError,OSError): pass\n        if len(candidates)!=1: raise ValueError(f\'Se necesita exactamente un estado v5 conectado (encontrados {len(candidates)}). Define RESUME_DIR explícitamente si hay varios.\')\n        source=candidates[0]\n    validate(source)\n    shutil.copytree(source,root,dirs_exist_ok=True)\n\ndef run(output_root=\'/kaggle/working/RESULTS_closure\',input_root=\'/kaggle/input\',resume_dir=\'\',session_hours=8.,reserve_minutes=15):\n    root=Path(output_root); inputs=Path(input_root)\n    initialize(root,inputs,resume_dir)\n    log=base.Monitor(root,60);log.start();error=None;paused=False;count=0;expected=240;state=None\n    started=time.monotonic(); deadline=started+max(0,session_hours*3600-reserve_minutes*60)\n    try:\n        log.log(\'PREVALIDACION v5: referencias, particiones congeladas y resultados reutilizados.\')\n        state,plans,count=validate(root); sci=state[\'scientific\'];expected=sci[\'expected\']\n        cfg={**base.DEFAULT,**{k:v for k,v in sci.items() if k in base.DEFAULT},\'workers\':0}\n        log.log(f\'PLAN {count}/{expected} completas verificadas; faltan {expected-count}. Presupuesto {session_hours} h; reserva {reserve_minutes} min.\')\n        base.summarize(root,cfg,SIGNATURE,log)\n        if count==expected: return\n        env=dict(python=sys.version,packages={k:importlib.metadata.version(k) for k in [\'torch\',\'torchvision\',\'timm\',\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\',\'ImageHash\']})\n        base.atomic_json(root/\'environment.json\',env)\n        origin=json.loads((root/\'origin_environment.json\').read_text())\n        for k in origin[\'packages\']:\n            if env[\'packages\'][k]!=origin[\'packages\'][k]: raise RuntimeError(f\'Entorno incompatible para reutilizar corridas: {k}={env["packages"][k]}, esperado {origin["packages"][k]}. Corregir antes de entrenar.\')\n        if not torch.cuda.is_available(): raise RuntimeError(\'GPU no disponible: activar GPU en Kaggle antes de ejecutar.\')\n        device=torch.device(\'cuda\');log.log(f\'GPU: {torch.cuda.get_device_name(0)}; usa una GPU; {torch.cuda.device_count()} visibles.\')\n        resolve_images(plans,inputs,log)\n        for name in sci[\'models\']:\n            if time.monotonic()>=deadline: raise base.PauseRun(\'Presupuesto agotado durante prevalidación.\')\n            log.log(f\'PREVALIDACION arquitectura {name}: forward y backward.\')\n            base.seed_all(42);m=base.build_model(name,cfg,device,pretrained=False)\n            with torch.autocast(device_type=\'cuda\',enabled=cfg[\'mixed_precision\']):\n                out=m(torch.zeros(2,3,224,224,device=device));loss=torch.nn.functional.cross_entropy(out,torch.tensor([0,1],device=device))\n            if out.shape!=(2,3) or not torch.isfinite(loss): raise ValueError(\'Salida de modelo inválida.\')\n            loss.backward();del m,out,loss;gc.collect();torch.cuda.empty_cache();log.log(f\'ARQUITECTURA OK: {name}\')\n        new_completed=0;training_start=time.monotonic()\n        for name in sci[\'models\']:\n            for domain,(d,plans_domain) in plans.items():\n                base.CLASSES=sci[\'domain_labels\'][domain]\n                for seed in sci[\'seeds\']:\n                    for strategy in [\'random\',\'group\']:\n                        parts=plans_domain[(strategy,seed)];rd=root/\'runs\'/f\'{domain}__{name}__{strategy}__{seed}\'\n                        if base.finished(rd,SIGNATURE,d.image_id.iloc[parts[2]]): continue\n                        base.train_one(d,parts,domain,name,strategy,seed,rd,cfg,SIGNATURE,device,log,deadline)\n                        count+=1;new_completed+=1\n                        eta=(time.monotonic()-training_start)/new_completed*(expected-count)/3600\n                        base.atomic_json(root/\'session_result.json\',dict(status=\'RUNNING\',completed=count,expected=expected,scientific_results=True,error=None))\n                        log.log(f\'PROGRESO GLOBAL {count}/{expected} ({100*count/expected:.1f}%); nuevas esta sesión={new_completed}; ETA aproximada={eta:.1f} h, varía según arquitectura.\')\n    except base.PauseRun as e:\n        paused=True;log.log(f\'PAUSA CONTROLADA: {e}\')\n    except (Exception,KeyboardInterrupt) as e:\n        error=e;log.log(f\'ERROR {type(e).__name__}: {e}\');(root/\'error.txt\').write_text(traceback.format_exc(),encoding=\'utf-8\')\n    finally:\n        try:\n            if state:\n                try: count=base.summarize(root,cfg,SIGNATURE,log)\n                except Exception as e:\n                    if error is None: error=e\n                    log.log(f\'ERROR al resumir: {e}\')\n            status=\'ERROR\' if error else (\'COMPLETE\' if count==expected else \'PAUSED\')\n            base.atomic_json(root/\'session_result.json\',dict(status=status,completed=count,expected=expected,scientific_results=error is None,error=str(error) if error else None))\n            export(root,log);log.log(f\'FIN {status}: {count}/{expected}.\')\n        finally: log.close()\n    if error: raise error\n',encoding="utf-8")
print("MOTOR v5 LISTO",flush=True)

## Ejecutar y conservar los resultados

El output informa imágenes comprobadas, inicio de corrida, lotes, época, validación, guardado y avance global. Un latido cada 60 segundos indica que el proceso responde; no demuestra avance del entrenamiento. La parada controlada ocurre a las 7 h 45 min con el presupuesto predeterminado. Puede requerir varias sesiones. Una cancelación forzada de Kaggle puede impedir la exportación; conservar el ZIP después de cada cierre controlado.

`COMPLETE` significa 240 corridas verificadas. `PAUSED` conserva lo realizado. `ERROR` explica el fallo y exporta lo disponible; no lo confunde con un estudio terminado. Los checkpoints guardan cada época completa; una época interrumpida se repite. No modifica semillas, épocas ni particiones al reanudar.


In [ ]:
import sys, importlib
if "/kaggle/working" not in sys.path: sys.path.insert(0,"/kaggle/working")
import engine_v5, runtime_v5
importlib.reload(engine_v5); importlib.reload(runtime_v5)
runtime_v5.run(session_hours=SESSION_HOURS,resume_dir=RESUME_DIR)
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/results_closure_v5.zip"))